# Inversión extranjera directa en México: de la API de datos.gob.mx a una tabla limpia

La Secretaría de Economía publica cada trimestre la inversión extranjera directa (IED) por estado, país de origen, sector y tipo de inversión. Los datos están en datos.gob.mx y se pueden consultar por API sin token. Este notebook descarga veinte años de cifras, documenta seis mañas del formato, baja el índice de precios con el que el notebook 02 pasa dólares corrientes a constantes y deja un snapshot limpio con el que trabaja el notebook 02.

**Fuente:** dataset `inversion_extranjera_directa` de la Secretaría de Economía en datos.gob.mx (licencia CC-BY 4.0), cifras en millones de dólares corrientes. Deflactor: CPI-U de Estados Unidos, serie `CPIAUCSL` de FRED (Federal Reserve Bank of St. Louis).

## 1. Preparación

Una sola dependencia de red (`requests`) y una de datos (`pandas`). La API rechaza el `User-Agent` por defecto de `requests`, así que la sesión usa uno de navegador.

In [1]:
import io
import json
import re
import time
from datetime import date
from pathlib import Path

import pandas as pd
import requests

API = "https://www.datos.gob.mx/api/3/action"
DATASET = "inversion_extranjera_directa"
UA = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/128.0 Safari/537.36"
)
SESION = requests.Session()
SESION.headers["User-Agent"] = UA

RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = RAIZ / "data"
DATA.mkdir(exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
print("Carpeta de datos:", DATA.relative_to(RAIZ))

Carpeta de datos: data


## 2. Tres puertas, una abierta

El dataset ofrece los CSV para descarga directa y la Secretaría de Economía publica Excel en su sitio. Ninguna de las dos responde a un programa: la descarga directa devuelve 403 y los Excel están detrás de un *challenge* de JavaScript. La API del datastore sí contesta, siempre que la petición parezca venir de un navegador.

In [2]:
paquete = SESION.get(f"{API}/package_show", params={"id": DATASET}, timeout=60)
paquete.raise_for_status()
meta = paquete.json()["result"]
url_csv_directo = meta["resources"][0]["url"]

print("Dataset:", meta["title"], "|", meta["organization"]["title"])
print("Última modificación:", meta["metadata_modified"][:10])
print("Recursos:", len(meta["resources"]))
print()
print("CSV directo              ->", requests.get(url_csv_directo, timeout=60).status_code)
print("API, User-Agent requests ->", requests.get(f"{API}/package_show", params={"id": DATASET}, timeout=60).status_code)
print("API, User-Agent navegador->", paquete.status_code)

Dataset: Inversión extranjera directa | Secretaría de Economía (SE)
Última modificación: 2026-08-28
Recursos: 20



CSV directo              -> 403
API, User-Agent requests -> 403
API, User-Agent navegador-> 200


Los veinte recursos siguen un patrón de nombre: un prefijo de lote (`b2026_01t`), `orig` o `act` según sean cifras originalmente publicadas o actualizadas, la dimensión principal (`ti` total, `po` país de origen, `ef` entidad federativa) y el cruce. El prefijo cambiará con cada lote, así que cada recurso se resuelve por su sufijo y nunca por su id.

In [3]:
def resolver_recursos(meta: dict) -> dict[str, str]:
    """Mapa sufijo -> resource_id. El sufijo ignora el prefijo de lote b####_##t_."""
    recursos = {}
    for r in meta["resources"]:
        nombre = r["url"].rsplit("/", 1)[-1].removesuffix(".csv")
        sufijo = re.sub(r"^b\d{4}_\d{2}t_", "", nombre)
        recursos[sufijo] = r["id"]
    return recursos


RECURSOS = resolver_recursos(meta)
pd.DataFrame(
    [(s, i, r["description"][:70]) for (s, i), r in zip(RECURSOS.items(), meta["resources"])],
    columns=["sufijo", "resource_id", "descripción"],
)

,sufijo,resource_id,descripción
0,orig_ti_tipodeinv,204b50c1-bc51-4a04-988c-b45e4deca161,Los datos incluyen los flujos de inversión ext...
1,orig_ti_destino,cf30cdeb-f5a4-4b6a-bd09-776ca1c71ccc,Los datos incluyen los flujos de inversión ext...
2,orig_ti_origen,9b53e7a7-12f3-4fde-9521-6586d7d436d2,Los datos incluyen los flujos de inversión ext...
3,orig_ti_sector,35fbafb9-d042-4698-9ced-345db4d4fca2,Los datos incluyen los flujos de inversión ext...
4,act_ti_tipodeinv,965c77a9-1807-4dd4-a6bf-4d32c12ad447,Los datos incluyen los flujos de inversión ext...
5,act_ti_destino,444d2c5f-84dc-4717-ae29-683f395dbf99,Los datos incluyen los flujos de inversión ext...
6,act_ti_origen,3fffc598-92a0-458e-b618-d20153401ed4,Los datos incluyen los flujos de inversión ext...
7,act_ti_sector,9a377082-27d9-4636-99ca-2cf6e28d72d6,Los datos incluyen los flujos de inversión ext...
8,orig_po_tipodeinv,43088d3d-3153-415c-9d1f-4acba4bd902f,Los datos incluyen los flujos de inversión ext...
9,orig_po_destino,13857bf0-24c0-435c-a762-805b3bf01100,Los datos incluyen los flujos de inversión ext...


## 3. Descarga paginada

El datastore devuelve como máximo 32,000 filas por llamada. La función pide páginas hasta cubrir `total`. Las cifras llegan como texto o número según la fila; se convierten a numérico y lo no convertible queda como nulo.

In [4]:
def descargar_tabla(sufijo: str, tamano_pagina: int = 32000) -> pd.DataFrame:
    """Descarga completa de un recurso del datastore, paginando por offset."""
    filas: list[dict] = []
    offset = 0
    while True:
        r = SESION.get(
            f"{API}/datastore_search",
            params={
                "resource_id": RECURSOS[sufijo],
                "limit": tamano_pagina,
                "offset": offset,
                "include_total": "true",
            },
            timeout=180,
        )
        r.raise_for_status()
        res = r.json()["result"]
        filas.extend(res["records"])
        offset += tamano_pagina
        if offset >= res["total"]:
            break
    df = pd.DataFrame(filas).drop(columns="_id")
    df["ied_mdd"] = pd.to_numeric(df["ied_mdd"], errors="coerce")
    return df


TABLAS = [
    "act_ti_tipodeinv",
    "act_ti_destino",
    "act_ti_origen",
    "act_ti_sector",
    "act_ef_tipodeinv",
    "orig_ti_tipodeinv",
    "orig_ti_destino",
    "orig_ti_origen",
]

crudas: dict[str, pd.DataFrame] = {}
for t in TABLAS:
    inicio = time.perf_counter()
    crudas[t] = descargar_tabla(t)
    print(f"{t:20s} {len(crudas[t]):>7,} filas  {time.perf_counter() - inicio:5.1f} s")

act_ti_tipodeinv         320 filas    0.2 s
act_ti_destino         2,640 filas    0.2 s


act_ti_origen         12,240 filas    0.2 s


act_ti_sector         31,680 filas    0.4 s


act_ef_tipodeinv      10,320 filas    0.3 s
orig_ti_tipodeinv        132 filas    0.1 s


orig_ti_destino        1,089 filas    0.1 s
orig_ti_origen         5,049 filas    0.2 s


## 4. Seis mañas del formato

Cada una con su evidencia. Son decisiones de publicación de la Secretaría que hay que conocer antes de sumar.

**Maña 1 · Los montos vienen acumulados dentro del año.** El valor del segundo trimestre incluye al primero, el del cuarto es el total anual.

In [5]:
ejemplo = crudas["act_ti_tipodeinv"]
ejemplo[(ejemplo.tipo_inversion == "Total general") & ejemplo.anio_trim.str.startswith("2025")]

,tipo_inversion,anio_trim,ied_mdd
76,Total general,2025-1,24412.609925
77,Total general,2025-2,38172.026805
78,Total general,2025-3,45908.439950
79,Total general,2025-4,40812.057628


**Maña 2 · Los nulos son cifras confidenciales.** La Secretaría publica "C" cuando un cruce identificaría a una empresa. En el datastore llegan como nulo. Se conservan como tales: rellenarlos con cero sesgaría cualquier suma.

In [6]:
pd.Series({t: int(df.ied_mdd.isna().sum()) for t, df in crudas.items()}, name="nulos (confidenciales)")

act_ti_tipodeinv        0
act_ti_destino          0
act_ti_origen        1856
act_ti_sector        4715
act_ef_tipodeinv        0
orig_ti_tipodeinv       0
orig_ti_destino         0
orig_ti_origen        818
Name: nulos (confidenciales), dtype: int64

**Maña 3 · El total cambia de nombre.** En casi todas las tablas es `Total general`; en `tipo_inversion` de la tabla por entidad es `Total`.

In [7]:
print("act_ti_tipodeinv:", sorted(crudas["act_ti_tipodeinv"].tipo_inversion.unique()))
print("act_ef_tipodeinv:", sorted(crudas["act_ef_tipodeinv"].tipo_inversion.unique()))

act_ti_tipodeinv: ['Cuentas entre compañías', 'Nuevas inversiones', 'Reinversión de utilidades', 'Total general']
act_ef_tipodeinv: ['Cuentas entre compañías', 'Nuevas inversiones', 'Reinversión de utilidades', 'Total']


**Maña 4 · Hay eñes rotas.** Algunos países traen `ń` (n con acento agudo) donde va `ñ`.

In [8]:
sorted(p for p in crudas["act_ti_origen"].pais_origen.unique() if "ń" in p)

['Espańa', 'Reino Unido de la Gran Bretańa e Irlanda del Norte']

**Maña 5 · Tres niveles del SCIAN en una columna.** `sector_subsector_rama` mezcla sectores (código de dos dígitos o rango como `31-33`), subsectores (tres) y ramas (cuatro). Sumar la columna completa cuenta cada peso tres veces.

In [9]:
etiquetas = pd.Series(crudas["act_ti_sector"].sector_subsector_rama.unique())
codigos = etiquetas.str.extract(r"^(\d{2}(?:-\d{2})?|\d{3,4})\s")[0]
print("Etiquetas:", len(etiquetas), "| sin código (totales):", int(codigos.isna().sum()))
print("Sectores (nivel 2):")
print(etiquetas[codigos.notna() & ~codigos.str.len().isin([3, 4])].to_string(index=False))

Etiquetas: 396 | sin código (totales): 1
Sectores (nivel 2):
11 Agricultura, cría y explotación de animales,...
                                        21 Minería
22 Generación, transmisión, distribución y come...
                                   23 Construcción
                   31-33 Industrias manufactureras
                          43 Comercio al por mayor
                          46 Comercio al por menor
       48-49 Transportes, correos y almacenamiento
                  51 Información en medios masivos
             52 Servicios financieros y de seguros
53 Servicios inmobiliarios y de alquiler de bie...
54 Servicios profesionales, científicos y técnicos
55 Dirección y administración de grupos empresa...
56 Servicios de apoyo a los negocios y manejo d...
                           61 Servicios educativos
      62 Servicios de salud y de asistencia social
71 Servicios de esparcimiento culturales y depo...
72 Servicios de alojamiento temporal y de prepa...
81 Otros servicios ex

**Maña 6 · Originales y actualizadas no coinciden.** La Secretaría revisa hacia atrás en cada informe trimestral. La tabla compara, año por año, el total que publicó primero con el que publica hoy. El análisis de largo plazo usa las actualizadas; las originales sirven para conciliar con los boletines, para medir cuánto se mueven las cifras recientes y para el primer trimestre de 2026, que todavía no tiene versión actualizada. El notebook 02 revisa qué hallazgos aguantan esas revisiones.

In [10]:
def cifra_anual(df: pd.DataFrame, tipo: str) -> pd.Series:
    """Acumulado del cuarto trimestre, es decir, el total del año, de un tipo de inversión."""
    cuarto = df[(df.tipo_inversion == tipo) & df.anio_trim.str.endswith("-4")]
    return cuarto.set_index(cuarto.anio_trim.str[:4].astype(int).rename("anio"))["ied_mdd"]


revision = pd.DataFrame({
    "original": cifra_anual(crudas["orig_ti_tipodeinv"], "Total general"),
    "actualizada": cifra_anual(crudas["act_ti_tipodeinv"], "Total general"),
}).dropna()
revision["diferencia"] = revision.actualizada - revision.original
revision["cambio_%"] = 100 * revision.diferencia / revision.original
print(f"Total anual {revision.index.min()} a {revision.index.max()}: revisado entre "
      f"{revision['cambio_%'].min():+.1f} % y {revision['cambio_%'].max():+.1f} % desde la primera cifra")
revision.round(1)

Total anual 2018 a 2025: revisado entre -2.7 % y +7.9 % desde la primera cifra


,original,actualizada,diferencia,cambio_%
anio,,,,
2018,31604.3,34111.0,2506.7,7.9
2019,32921.2,34622.0,1700.7,5.2
2020,29079.4,28280.2,-799.2,-2.7
2021,31621.2,33847.7,2226.5,7.0
2022,35291.6,36420.5,1128.9,3.2
2023,36058.0,36477.6,419.6,1.2
2024,36872.4,37939.3,1066.8,2.9
2025,40870.8,40812.1,-58.7,-0.1


## 5. Normalizar y desacumular

`normalizar` unifica etiquetas, corrige eñes, separa año y trimestre, y descompone el SCIAN. `desacumular` convierte el acumulado en flujo trimestral: el primer trimestre es su propio flujo; los demás restan el trimestre anterior del mismo año. Si el anterior es confidencial, el flujo queda nulo en vez de inventarse.

In [11]:
DIMENSIONES = ["tipo_inversion", "entidad_federativa", "pais_origen", "sector_subsector_rama"]


def normalizar(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in DIMENSIONES:
        if col in df:
            df[col] = df[col].str.strip().str.replace("ń", "ñ", regex=False)
            df.loc[df[col].isin(["Total", "Total general"]), col] = "Total general"
    partes = df["anio_trim"].str.split("-", expand=True)
    df["anio"] = partes[0].astype(int)
    df["trimestre"] = partes[1].astype(int)
    if "sector_subsector_rama" in df:
        codigo = df["sector_subsector_rama"].str.extract(r"^(\d{2}(?:-\d{2})?|\d{3,4})\s")[0]
        df["scian_codigo"] = codigo
        df["scian_nivel"] = codigo.map(
            lambda c: None if pd.isna(c) else "sector" if ("-" in c or len(c) == 2) else "subsector" if len(c) == 3 else "rama"
        )
        df["sector"] = df["sector_subsector_rama"].str.replace(r"^(\d{2}(?:-\d{2})?|\d{3,4})\s+", "", regex=True)
    df["confidencial"] = df["ied_mdd"].isna()
    return df.drop(columns="anio_trim")


def desacumular(df: pd.DataFrame) -> pd.DataFrame:
    claves = [c for c in DIMENSIONES if c in df]
    df = df.sort_values(claves + ["anio", "trimestre"]).reset_index(drop=True)
    previo = df.groupby(claves + ["anio"])["ied_mdd"].shift(1)
    primero = df["trimestre"].eq(1)
    df["ied_acumulado_mdd"] = df["ied_mdd"]
    df["flujo_mdd"] = df["ied_mdd"] - previo.where(~primero, 0.0)
    return df.drop(columns="ied_mdd")


limpias = {t: desacumular(normalizar(df)) for t, df in crudas.items()}
limpias["act_ti_tipodeinv"].query("tipo_inversion == 'Total general' and anio == 2025")

,tipo_inversion,anio,trimestre,confidencial,ied_acumulado_mdd,flujo_mdd
316,Total general,2025,1,False,24412.609925,24412.609925
317,Total general,2025,2,False,38172.026805,13759.416880
318,Total general,2025,3,False,45908.439950,7736.413145
319,Total general,2025,4,False,40812.057628,-5096.382322


## 6. Validaciones

Tres comprobaciones. Dos son aserciones: si fallan, el notebook se detiene y el snapshot no se escribe. La tercera cuantifica una brecha que no debe forzarse.

**V1 · La suma de los estados es el total nacional**, trimestre a trimestre, en cifras actualizadas.

In [12]:
d = limpias["act_ti_destino"]
por_estado = d[d.entidad_federativa != "Total general"].groupby(["anio", "trimestre"])["flujo_mdd"].sum()
total = d[d.entidad_federativa == "Total general"].set_index(["anio", "trimestre"])["flujo_mdd"]
dif = (por_estado - total).abs()
anio_max, trim_max = (int(x) for x in dif.idxmax())
print(f"Diferencia máxima estados vs total: {dif.max():.4f} mdd, en {anio_max}-{trim_max}")
assert dif.max() < 1.0, "Los estados no suman el total nacional"

Diferencia máxima estados vs total: 0.0000 mdd, en 2025-2


**V2 · Países: cuánto queda en confidencial.** La suma de países publicados no tiene por qué cerrar contra el total, porque los confidenciales no se publican. Se cuantifica la brecha por año en vez de forzarla.

In [13]:
p = limpias["act_ti_origen"]
brecha = (
    p[p.pais_origen == "Total general"].groupby("anio")["flujo_mdd"].sum()
    - p[p.pais_origen != "Total general"].groupby("anio")["flujo_mdd"].sum()
)
confidenciales = p[p.pais_origen != "Total general"].groupby("anio")["confidencial"].sum()
pd.DataFrame({"brecha_mdd": brecha.round(1), "celdas_confidenciales": confidenciales})

,brecha_mdd,celdas_confidenciales
anio,,
2006,11.8,88
2007,664.8,99
2008,72.7,84
2009,-67.0,66
2010,-109.3,88
2011,133.8,86
2012,10.4,83
2013,51.6,79
2014,41.8,99


**V3 · Las cifras originales cuadran con lo que publicó la Secretaría de Economía:** con los boletines del 24 de febrero de 2026 (año 2025) y del 25 de mayo de 2026 (primer trimestre de 2026), y con el Cuadro 1 del [informe estadístico del cuarto trimestre de 2025](https://gaceta.diputados.gob.mx/PDF/66/2026/abr/InvExt-20260414.pdf), que da la cifra originalmente publicada de cada año desde 2018 por tipo de inversión.

In [14]:
o = limpias["orig_ti_tipodeinv"].set_index(["tipo_inversion", "anio", "trimestre"])["ied_acumulado_mdd"]
boletines = {
    ("Total general", 2025, 4): 40_871,
    ("Total general", 2026, 1): 23_591,
    ("Nuevas inversiones", 2026, 1): 1_705,
    ("Reinversión de utilidades", 2026, 1): 22_222,
}
for clave, publicado in boletines.items():
    calculado = o[clave]
    print(f"{clave[0]:26s} {clave[1]}-{clave[2]}  datastore {calculado:10,.1f}  boletín {publicado:7,}")
    assert abs(round(calculado) - publicado) <= 1, f"No cuadra: {clave}"

# Cuadro 1 del informe del cuarto trimestre de 2025: cifra original de 2018 a 2025, con un decimal.
CUADRO_1 = {
    "Total general": [31_604.3, 32_921.2, 29_079.4, 31_621.2, 35_291.6, 36_058.0, 36_872.4, 40_870.8],
    "Nuevas inversiones": [11_468.3, 12_826.8, 6_408.5, 13_825.3, 16_993.1, 4_817.4, 3_168.5, 7_377.4],
    "Reinversión de utilidades": [12_251.2, 17_481.7, 16_095.7, 12_213.0, 16_027.8, 26_630.6, 28_710.3, 27_649.3],
    "Cuentas entre compañías": [7_884.8, 2_612.7, 6_575.3, 5_582.9, 2_270.7, 4_610.0, 4_993.6, 5_844.0],
}
for tipo, cifras in CUADRO_1.items():
    for anio, publicado in zip(range(2018, 2026), cifras):
        assert abs(o[(tipo, anio, 4)] - publicado) <= 0.05, f"No cuadra con el Cuadro 1: {tipo} {anio}"
print(f"Cuadro 1 del informe del cuarto trimestre de 2025: las {sum(map(len, CUADRO_1.values()))} cifras cuadran al decimal")

Total general              2025-4  datastore   40,870.8  boletín  40,871
Total general              2026-1  datastore   23,590.6  boletín  23,591
Nuevas inversiones         2026-1  datastore    1,705.2  boletín   1,705
Reinversión de utilidades  2026-1  datastore   22,221.5  boletín  22,222
Cuadro 1 del informe del cuarto trimestre de 2025: las 32 cifras cuadran al decimal


## 7. Deflactor: el CPI-U de Estados Unidos

Las cifras de la Secretaría están en dólares corrientes. Para comparar años sin la inflación, el notebook 02 las pasa a dólares constantes con el índice de precios al consumidor de Estados Unidos (CPI-U, serie `CPIAUCSL` de FRED, mensual y desestacionalizada), promediado por año. Solo se guardan años con sus doce meses.

FRED se porta al revés que datos.gob.mx: al `User-Agent` de navegador no le contesta, y al de `requests` le responde al instante. Por eso esta descarga no usa la sesión de arriba.

In [15]:
FRED_CSV = "https://fred.stlouisfed.org/graph/fredgraph.csv"
try:
    prueba = requests.get(FRED_CSV, params={"id": "CPIAUCSL"}, headers={"User-Agent": UA}, timeout=10)
    print("FRED, User-Agent navegador ->", prueba.status_code)
except requests.RequestException as e:
    print("FRED, User-Agent navegador -> sin respuesta en 10 s", f"({type(e).__name__})")
respuesta = requests.get(FRED_CSV, params={"id": "CPIAUCSL"}, timeout=60)
respuesta.raise_for_status()
print("FRED, User-Agent requests  ->", respuesta.status_code)

cpi_mensual = pd.read_csv(io.StringIO(respuesta.text), parse_dates=["observation_date"])
ULTIMO_MES = cpi_mensual.observation_date.max()
cpi = (
    cpi_mensual.assign(anio=cpi_mensual.observation_date.dt.year)
    .groupby("anio")["CPIAUCSL"]
    .agg(cpi_promedio="mean", meses="size")
    .reset_index()
)
cpi = cpi[(cpi.meses == 12) & (cpi.anio >= 2006)].reset_index(drop=True)
print(f"CPIAUCSL: {len(cpi_mensual):,} meses, el último de {ULTIMO_MES:%Y-%m}; "
      f"años completos desde 2006: {cpi.anio.min()} a {cpi.anio.max()}")
cpi.tail(3)

FRED, User-Agent navegador -> sin respuesta en 10 s (ReadTimeout)
FRED, User-Agent requests  -> 200
CPIAUCSL: 956 meses, el último de 2026-08; años completos desde 2006: 2006 a 2025


,anio,cpi_promedio,meses
17,2023,304.703000,12
18,2024,313.698167,12
19,2025,321.961909,12


## 8. Snapshot

Cada tabla limpia se guarda como CSV. El manifiesto registra la fecha de consulta, el lote de la Secretaría, la URL de origen y el último trimestre de cada tabla, y lo mismo para el deflactor. El notebook 02 lee solo estos archivos: no hace llamadas a la red.

In [16]:
URL_POR_ID = {r["id"]: r["url"] for r in meta["resources"]}
LOTE = re.match(r"b\d{4}_\d{2}t", URL_POR_ID[RECURSOS["act_ti_destino"]].rsplit("/", 1)[-1]).group(0)


def ultimo_trimestre(df: pd.DataFrame) -> str:
    anio = int(df.anio.max())
    return f"{anio}-T{int(df.loc[df.anio == anio, 'trimestre'].max())}"


manifiesto = {
    "fecha_consulta": date.today().isoformat(),
    "dataset": DATASET,
    "url_dataset": f"{API}/package_show?id={DATASET}",
    "lote": LOTE,
    "metadata_modified": meta["metadata_modified"],
    "licencia": meta["license_title"],
    "user_agent": UA,
    "ultimo_trimestre": max(ultimo_trimestre(df) for df in limpias.values()),
    "tablas": {},
}
for t, df in limpias.items():
    ruta = DATA / f"{t}.csv"
    df.to_csv(ruta, index=False)
    manifiesto["tablas"][t] = {
        "resource_id": RECURSOS[t],
        "url_api": f"{API}/datastore_search?resource_id={RECURSOS[t]}",
        "url_csv": URL_POR_ID[RECURSOS[t]],
        "filas": int(len(df)),
        "confidenciales": int(df.confidencial.sum()),
        "anio_min": int(df.anio.min()),
        "anio_max": int(df.anio.max()),
        "ultimo_trimestre": ultimo_trimestre(df),
    }
cpi.to_csv(DATA / "cpi_eeuu.csv", index=False)
manifiesto["deflactor"] = {
    "archivo": "cpi_eeuu.csv",
    "serie": "CPIAUCSL",
    "descripcion": "Consumer Price Index for All Urban Consumers: All Items in U.S. City Average; "
                   "1982-1984 = 100; mensual, desestacionalizado",
    "fuente": "U.S. Bureau of Labor Statistics vía FRED, Federal Reserve Bank of St. Louis",
    "url": "https://fred.stlouisfed.org/series/CPIAUCSL",
    "url_descarga": f"{FRED_CSV}?id=CPIAUCSL",
    "fecha_consulta": date.today().isoformat(),
    "agregacion": "promedio anual de los doce meses; solo años completos",
    "ultimo_mes": f"{ULTIMO_MES:%Y-%m}",
    "anio_min": int(cpi.anio.min()),
    "anio_max": int(cpi.anio.max()),
}
(DATA / "manifest.json").write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2) + "\n")
print(f"Lote {LOTE} · último trimestre publicado: {manifiesto['ultimo_trimestre']}")
pd.DataFrame(manifiesto["tablas"]).T[["filas", "confidenciales", "anio_min", "anio_max", "ultimo_trimestre"]]

Lote b2026_01t · último trimestre publicado: 2026-T1


,filas,confidenciales,anio_min,anio_max,ultimo_trimestre
act_ti_tipodeinv,320,0,2006,2025,2025-T4
act_ti_destino,2640,0,2006,2025,2025-T4
act_ti_origen,12240,1856,2006,2025,2025-T4
act_ti_sector,31680,4715,2006,2025,2025-T4
act_ef_tipodeinv,10320,0,2006,2025,2025-T4
orig_ti_tipodeinv,132,0,2018,2026,2026-T1
orig_ti_destino,1089,0,2018,2026,2026-T1
orig_ti_origen,5049,818,2018,2026,2026-T1


## 9. Qué se puede afirmar con estos datos, y qué no

- **Sí:** flujos anuales y trimestrales por estado, país, sector y tipo de inversión, de 2006 a 2025 en cifras actualizadas y hasta el primer trimestre de 2026 en originales.
- **Con cuidado, el estado.** La descripción del dataset no dice cómo se asigna la IED a cada estado. Lo dicen el apéndice metodológico del [informe estadístico del cuarto trimestre de 2025](https://gaceta.diputados.gob.mx/PDF/66/2026/abr/InvExt-20260414.pdf) (sección 3.4) y la [síntesis metodológica de la Secretaría](https://www.gob.mx/cms/uploads/attachment/file/274849/Sintesis_metodologica_IED.pdf) (sección 4): desde 2015 la Secretaría usa la localización operativa y la estructura corporativa de cada empresa, con el destino que la empresa reporta o, si no lo tiene, con un reparto que estima la propia Secretaría. Las tablas no dicen qué parte de cada estado viene de cada criterio.
- **Con cuidado, el país.** El mismo apéndice clasifica el país "en función del origen de los recursos". La [síntesis metodológica de la Secretaría](https://www.gob.mx/cms/uploads/attachment/file/274849/Sintesis_metodologica_IED.pdf) precisa que eso identifica al país donde reside el inversionista directo, que no siempre es el país de quien controla la inversión en última instancia.
- **Con cuidado, los años recientes.** Todas las cifras son preliminares: la Secretaría las corrige en cada informe y las correcciones más grandes caen en los trimestres recientes (sección 3.8 del mismo apéndice). La maña 6 mide cuánto se movió cada total anual.
- **No:** sumar `sector_subsector_rama` sin filtrar nivel, rellenar confidenciales con cero, o comparar cifras originales con actualizadas como si fueran la misma serie.